# 02 · Finding Market Regimes with a Hidden Markov Model
**Week 2 of _Can a Neural Network See Market Regimes Coming?_**

In notebook 01 we saw that markets switch between calm and violent periods. Here we let a
**Hidden Markov Model (HMM)** find those periods ("regimes") automatically.

### The idea in plain words
- Each day the market is in one of a few **hidden states** (we choose 3). We never observe the state directly.
- What we *do* observe each day is the return and the volatility. Each state produces observations with its own
  typical pattern (calm = small returns and low volatility, crisis = big swings and high volatility).
- States are **sticky**: a **transition matrix** gives the probability of moving from one state to another tomorrow.

The HMM learns the typical pattern of each state and the transition probabilities from data, then tells us
which state each day most likely belonged to.

In [ ]:
!pip -q install yfinance hmmlearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from hmmlearn.hmm import GaussianHMM

plt.rcParams["figure.figsize"] = (11, 4)

## 1. Data and features (same functions as notebook 01)

In [ ]:
def load_close(ticker, start="2008-01-01"):
    """Download daily closing prices from Yahoo Finance as a pandas Series."""
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    close = df["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance returns a 1-column DataFrame
        close = close.iloc[:, 0]
    return close.dropna()

def make_features(close, vol_window=20):
    """Daily log return + rolling annualised volatility."""
    df = pd.DataFrame({"close": close})
    df["ret"] = np.log(df["close"]).diff()                                   # log return
    df["vol"] = df["ret"].rolling(vol_window).std() * np.sqrt(252)           # annualised volatility
    return df.dropna()

nifty = load_close("^NSEI")
df = make_features(nifty)
print(df.shape)

## 2. Train/test split: avoiding *lookahead bias*
If we fit the HMM on **all** the data, it has "seen the future" (e.g. it already knows the 2020 crash happened).
Any strategy built on that would look amazing on paper and fail in reality.

So we fit only on data **before 2019** and treat 2019 onwards as unseen test data.
We also standardise features (subtract mean, divide by std) using **training** statistics only.

In [ ]:
REGIME_NAMES = ["Calm", "Turbulent", "Crisis"]
SPLIT_DATE = "2019-01-01"        # everything before = training data, from here on = test data

def fit_hmm(df, n_states=3, seed=42):
    """Fit an HMM on TRAINING data only. Returns (model, mean, std, order)."""
    X = df[["ret", "vol"]].values
    train = df.index < SPLIT_DATE
    mean, std = X[train].mean(axis=0), X[train].std(axis=0)
    Xs = (X - mean) / std
    model = GaussianHMM(n_components=n_states, covariance_type="full",
                        n_iter=500, random_state=seed)
    model.fit(Xs[train])
    order = np.argsort(model.means_[:, 1])        # sort states by volatility: low -> high
    return model, mean, std, order

def hmm_labels(df, model, mean, std, order):
    """Most likely regime for every day (Viterbi), renamed so 0=Calm, 1=Turbulent, 2=Crisis."""
    Xs = (df[["ret", "vol"]].values - mean) / std
    raw = model.predict(Xs)
    rename = {old: new for new, old in enumerate(order)}
    return np.array([rename[s] for s in raw])

model, mean, std, order = fit_hmm(df)
print("Converged:", model.monitor_.converged)
df["regime"] = hmm_labels(df, model, mean, std, order)
df["regime_name"] = [REGIME_NAMES[i] for i in df["regime"]]
df["regime_name"].value_counts()

## 3. What does each regime look like?

In [ ]:
summary = df.groupby("regime_name").agg(
    days=("ret", "size"),
    mean_annual_return=("ret", lambda r: r.mean() * 252),
    annual_vol=("ret", lambda r: r.std() * np.sqrt(252)),
).reindex(REGIME_NAMES)
summary["share_of_days"] = summary["days"] / summary["days"].sum()
summary.style.format({"mean_annual_return": "{:.1%}", "annual_vol": "{:.1%}", "share_of_days": "{:.1%}"})

**What to look for:** volatility should rise from Calm → Turbulent → Crisis (we sorted the states that way).
Returns are usually highest in Calm and lowest (often negative) in Crisis.

## 4. The transition matrix: how sticky are regimes?

In [ ]:
T = model.transmat_[np.ix_(order, order)]     # reorder rows/cols to Calm, Turbulent, Crisis
T = pd.DataFrame(T, index=REGIME_NAMES, columns=REGIME_NAMES)
print("P(tomorrow's regime | today's regime):")
print(T.round(3))

# If you stay in a regime with probability p each day, the average stay is 1 / (1 - p) days.
print("\nExpected length of a stay (days):")
print((1 / (1 - np.diag(T.values))).round(1))

## 5. Regimes on the price chart

In [ ]:
colors = {"Calm": "tab:green", "Turbulent": "tab:orange", "Crisis": "tab:red"}
fig, ax = plt.subplots(figsize=(12, 5))
for name, c in colors.items():
    part = df[df["regime_name"] == name]
    ax.scatter(part.index, part["close"], s=3, color=c, label=name)
ax.axvline(pd.Timestamp(SPLIT_DATE), color="black", ls="--", label="train | test")
ax.set_yscale("log"); ax.set_title("Nifty 50 coloured by HMM regime"); ax.legend()
plt.show()

## 6. Why 3 states? Comparing models with BIC
More states always fit the training data better, so we need a penalty for complexity.
**BIC** (Bayesian Information Criterion) = −2 × log-likelihood + (number of parameters) × ln(number of days).
**Lower BIC is better.**

In [ ]:
train = df.index < SPLIT_DATE
Xs_train = ((df[["ret", "vol"]].values - mean) / std)[train]
n, d = Xs_train.shape
for k in [2, 3, 4, 5]:
    m = GaussianHMM(n_components=k, covariance_type="full", n_iter=500, random_state=42).fit(Xs_train)
    n_params = (k - 1) + k * (k - 1) + k * d + k * d * (d + 1) / 2   # start + transitions + means + covariances
    bic = -2 * m.score(Xs_train) + n_params * np.log(n)
    print(f"{k} states: BIC = {bic:,.0f}")

BIC often prefers more than 3 states. We still use 3 because they are **interpretable** and match how
investors talk about markets. That is a judgement call worth discussing with mentees.

## Limitations to remember
- `model.predict` uses the **Viterbi** algorithm, which looks at the *whole* sequence, including days after the
  one it labels. These labels are fine as a description or as training targets, but you could not have known them
  on the day itself. Notebook 04 shows how to get regimes using only past data.
- Results depend on the random seed and the split date. Try changing them.

## Check your understanding
1. What is "hidden" in a Hidden Markov Model, and what is observed?
2. What does one row of the transition matrix mean? Why do its entries add up to 1?
3. Why must the HMM be fitted on training data only?
4. How do we decide which state is "Calm" and which is "Crisis"? (Hint: look at `order`.)
5. If P(Calm → Calm) = 0.98, how long does a calm spell last on average?